# RoadSoS - Physics-Informed Neural Network
**IITM Hackathon 2026**

Skid prediction + HIC15 + BrIC brain injury estimation.

- **Physics loss 1**: Two-wheeler tire-road ODE (velocity + tilt dynamics)
- **Physics loss 2**: Kelvin-Voigt brain spring-mass-damper (3-axis)

Run cells top to bottom.

In [1]:
!pip install torch numpy matplotlib pandas scipy streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 59.7 MB/s eta 0:00:0000:010:01


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, time
warnings.filterwarnings("ignore")

plt.style.use("dark_background")
DARK_ACCENT = "#00E5FF"
WARN_ACCENT = "#FFD600"
CRIT_ACCENT = "#FF1744"
SAFE_ACCENT = "#00E676"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [6]:
# Physical constants

M_TOTAL   = 225.0   # kg (rider 75 + bike 150)
G         = 9.81    # m/s2
WHEELBASE = 1.35    # m

MU_DRY = 0.75
MU_WET = 0.45
MU_OIL = 0.15

# HIC thresholds (ECE 22.06 helmet standard)
HIC_LOW  = 700
HIC_HIGH = 1000

# BrIC critical angular velocities (rad/s) - NHTSA cadaver study
BRIC_X = 66.3   # sagittal / pitch
BRIC_Y = 56.5   # coronal  / roll
BRIC_Z = 42.2   # transverse / yaw

# Kelvin-Voigt brain biomechanics
M_BRAIN = 1.4     # kg
K_BRAIN = 2.1e4   # N/m
C_BRAIN = 85.0    # N.s/m  (natural freq ~ 19.5 Hz, concussion-risk band)

FS_NORMAL = 100   # Hz
FS_CRASH  = 1000  # Hz (HIC needs >= 1kHz)

print(f"Brain natural freq: {(K_BRAIN/M_BRAIN)**0.5/(2*3.14159):.1f} Hz")

Brain natural freq: 19.5 Hz


In [ ]:
# PINN Architecture

class RoadSoSPINN(nn.Module):  # Initialize the Model
    """
    Input  (7): [t, ax, ay, az, gx, gy, gz]
    Output (6): [v, theta, mu_eff, x_brain, y_brain, z_brain]

    tanh: smooth higher-order derivatives required for physics residuals.
    Xavier init: stable variance through deep tanh stack.
    sigmoid on mu_eff: physical constraint to (0, 1).
    """
    def __init__(self, hidden_layers=6, hidden_width=128):
        super().__init__()
        dims = [7] + [hidden_width]*hidden_layers + [6]
        layers = []
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

    def get_outputs(self, x):
        o = self.forward(x)
        return {
            "v":       o[:, 0:1],
            "theta":   o[:, 1:2],
            "mu_eff":  torch.sigmoid(o[:, 2:3]),
            "x_brain": o[:, 3:4],
            "y_brain": o[:, 4:5],
            "z_brain": o[:, 5:6],
        }


class IMUNormaliser:
    def __init__(self): self.mean = None; self.std = None
    def fit_transform(self, X):
        self.mean = X.mean(0); self.std = X.std(0) + 1e-8
        return (X - self.mean) / self.std
    def transform(self, X): return (X - self.mean) / self.std
    def inverse_transform(self, X): return X * self.std + self.mean


_m = RoadSoSPINN().to(DEVICE)
# Create leaf tensor directly on device; .to(DEVICE) after requires_grad=True
# produces a non-leaf node whose .grad stays None after backward.
_x = torch.randn(8, 7, device=DEVICE, requires_grad=True)
_o = _m(_x); _o.sum().backward()
print(f"Parameters : {sum(p.numel() for p in _m.parameters()):,}")
print(f"Output     : {_o.shape}  (expect [8, 6])")
print(f"Grads OK   : {_x.grad is not None}")
del _m, _x, _o

In [8]:
# Synthetic Data Generation

def simulate_normal(duration=10.0, fs=FS_NORMAL, v0=11.1, seed=0):
    np.random.seed(seed)
    t = np.linspace(0, duration, int(duration*fs)); n = len(t)
    v     = v0 + 0.5*np.sin(0.3*t) + 0.1*np.random.randn(n)
    theta = 0.05*np.sin(0.2*t) + 0.01*np.random.randn(n)
    mu    = np.clip(MU_DRY + 0.02*np.random.randn(n), 0.6, 0.9)
    ax    = np.diff(v, prepend=v[0])*fs + 0.10*np.random.randn(n)
    ay    = v*np.gradient(theta, 1/fs)  + 0.10*np.random.randn(n)
    az    = G*np.cos(theta)             + 0.05*np.random.randn(n)
    gx    = np.gradient(theta, 1/fs)   + 0.02*np.random.randn(n)
    gy    = 0.05*np.sin(0.10*t)        + 0.01*np.random.randn(n)
    gz    = 0.03*np.cos(0.15*t)        + 0.01*np.random.randn(n)
    return t, ax, ay, az, gx, gy, gz, v, theta, mu, np.zeros(n), np.zeros(n,int)


def simulate_oil_patch(duration=12.0, fs=FS_NORMAL, v0=13.9, patch_start=4.0, seed=1):
    np.random.seed(seed)
    t = np.linspace(0, duration, int(duration*fs)); n = len(t)
    pi  = int(patch_start*fs)
    dec = np.exp(-np.arange(n-pi) / (2.0*fs))
    mu  = np.ones(n)*MU_DRY
    mu[pi:] = MU_OIL + (MU_DRY-MU_OIL)*dec[:n-pi]
    mu  = np.clip(mu + 0.01*np.random.randn(n), 0.05, 0.9)
    v   = v0 + 0.8*np.sin(0.3*t)
    v[pi:] -= 2.5*(1-dec[:n-pi]); v = np.clip(v, 0, 30)
    theta = 0.05*np.sin(0.2*t); theta[pi:] += 0.3*(1-dec[:n-pi])
    theta += 0.01*np.random.randn(n)
    ax  = np.diff(v, prepend=v[0])*fs + 0.15*np.random.randn(n)
    ay  = v*np.gradient(theta, 1/fs)  + 0.15*np.random.randn(n)
    az  = G*np.cos(theta)             + 0.08*np.random.randn(n)
    gx  = np.gradient(theta, 1/fs)   + 0.03*np.random.randn(n)
    gy  = 0.05*np.sin(0.10*t)        + 0.02*np.random.randn(n)
    gz  = v/WHEELBASE*np.tan(theta)  + 0.02*np.random.randn(n)
    P_sk = (np.clip((MU_WET-mu)/MU_WET,0,1)>0.3).astype(float)
    return t, ax, ay, az, gx, gy, gz, v, theta, mu, P_sk, np.ones(n,int)


def simulate_crash(duration=2.0, fs=FS_CRASH, v0=16.7, impact_t=0.5, seed=2):
    np.random.seed(seed)
    t = np.linspace(0, duration, int(duration*fs)); n = len(t)
    ii = int(impact_t*fs); sw = int(0.10*fs)
    v  = np.ones(n)*v0
    v[ii:ii+sw] = np.linspace(v0, 0, sw); v[ii+sw:] = 0.0
    theta = np.zeros(n)
    theta[ii:] = np.clip(np.linspace(0, np.pi/2*1.2, n-ii), 0, np.pi/2)
    mu = np.ones(n)*MU_DRY; mu[ii:] = MU_OIL
    pw = int(0.010*fs); pulse = np.zeros(n)
    if ii+pw < n:
        pulse[ii:ii+pw] = 10*G*np.sin(np.linspace(0, np.pi, pw))
    ax = np.diff(v, prepend=v[0])*fs + pulse + 0.2*np.random.randn(n)
    ay = 3*G*np.sin(theta) + 0.3*np.random.randn(n)
    az = G*np.cos(theta) - 0.3*pulse + 0.2*np.random.randn(n)
    ff_s = ii+pw; ff_e = ff_s+int(0.05*fs)
    if ff_e < n: az[ff_s:ff_e] *= 0.1   # free-fall
    gx = np.gradient(theta, 1/fs)
    if ii+pw < n: gx[ii:ii+pw] += 25*np.sin(np.linspace(0, np.pi, pw))
    gy = 10*np.sin(np.linspace(0,3*np.pi,n))*(t>impact_t)+0.5*np.random.randn(n)
    gz =  8*np.sin(np.linspace(0,2*np.pi,n))*(t>impact_t)+0.3*np.random.randn(n)
    P_sk = (t >= impact_t).astype(float)
    return t, ax, ay, az, gx, gy, gz, v, theta, mu, P_sk, 2*np.ones(n,int)


def build_dataset():
    all_X, all_Y, all_Psk, all_sc = [], [], [], []
    for seed in range(4):
        t,ax,ay,az,gx,gy,gz,v,th,mu,psk,sc = simulate_normal(duration=15.0,seed=seed)
        all_X.append(np.c_[t,ax,ay,az,gx,gy,gz]); all_Y.append(np.c_[v,th,mu])
        all_Psk.append(psk); all_sc.append(sc)
    for seed in range(4):
        t,ax,ay,az,gx,gy,gz,v,th,mu,psk,sc = simulate_oil_patch(seed=seed+10)
        all_X.append(np.c_[t,ax,ay,az,gx,gy,gz]); all_Y.append(np.c_[v,th,mu])
        all_Psk.append(psk); all_sc.append(sc)
    for seed in range(4):
        t,ax,ay,az,gx,gy,gz,v,th,mu,psk,sc = simulate_crash(seed=seed+20)
        all_X.append(np.c_[t,ax,ay,az,gx,gy,gz]); all_Y.append(np.c_[v,th,mu])
        all_Psk.append(psk); all_sc.append(sc)
    X = np.vstack(all_X); Y = np.vstack(all_Y)
    meta = {"P_skid":np.concatenate(all_Psk),"scenario":np.concatenate(all_sc)}
    print(f"Dataset: {X.shape[0]:,} samples")
    for k,lbl in [(0,"Normal"),(1,"Oil patch"),(2,"Crash")]:
        print(f"  {lbl:10s}: {(meta['scenario']==k).sum():,}")
    return X, Y, meta

X_raw, Y_raw, meta = build_dataset()

Dataset: 18,800 samples
  Normal    : 6,000
  Oil patch : 4,800
  Crash     : 8,000


In [ ]:
# HIC15 and BrIC

def compute_hic15(ax, ay, az, fs, window_ms=15):
    a_res = np.sqrt(ax**2 + ay**2 + az**2) / G
    dt    = 1.0 / fs
    max_w = max(2, int(window_ms * 1e-3 / dt))
    hic = 0.0; best = (0, 0)
    cs = np.cumsum(a_res)  # computed once; identical every iteration
    for w in range(2, max_w+1):
        means = (cs[w:] - cs[:-w]) / w
        vals  = w * dt * (np.maximum(means, 0) ** 2.5)
        i     = np.argmax(vals)
        if vals[i] > hic: hic = vals[i]; best = (i, i+w)
    return hic, best

def compute_bric(gx, gy, gz):
    return np.sqrt(
        (np.max(np.abs(gx))/BRIC_X)**2 +
        (np.max(np.abs(gy))/BRIC_Y)**2 +
        (np.max(np.abs(gz))/BRIC_Z)**2)

def injury_label(hic, bric):
    score = min(0.6*hic/HIC_HIGH + 0.4*bric/1.0, 1.0)
    if score < 0.3: return "LOW",      SAFE_ACCENT, score
    if score < 0.7: return "MODERATE", WARN_ACCENT, score
    return             "SEVERE",       CRIT_ACCENT, score

# Test
_,ax_c,ay_c,az_c,gx_c,gy_c,gz_c,*_ = simulate_crash(seed=99)
h,_ = compute_hic15(ax_c,ay_c,az_c,FS_CRASH)
b   = compute_bric(gx_c,gy_c,gz_c)
lbl,_,_ = injury_label(h,b)
print(f"Crash  HIC15={h:.0f}  BrIC={b:.3f}  [{lbl}]")

_,ax_n,ay_n,az_n,gx_n,gy_n,gz_n,*_ = simulate_normal(seed=99)
h2,_ = compute_hic15(ax_n,ay_n,az_n,FS_NORMAL)
b2   = compute_bric(gx_n,gy_n,gz_n)
lbl2,_,_ = injury_label(h2,b2)
print(f"Normal HIC15={h2:.1f}  BrIC={b2:.3f}  [{lbl2}]")

In [ ]:
# Physics Residuals and Loss Function

def residual_vehicle(outs, x):
    """
    Two-wheeler ODEs via autograd:
      (1) M*dv/dt + mu_eff*M*G = 0       (friction deceleration)
      (2) dtheta/dt - v*sin(theta)/L = 0  (tilt kinematic)
    x must have requires_grad=True.
    Residuals are divided by their characteristic scales so both terms
    stay O(1) and the lambda weights stay interpretable.
    """
    dv_dt     = grad(outs["v"].sum(),     x, create_graph=True)[0][:, 0:1]
    dtheta_dt = grad(outs["theta"].sum(), x, create_graph=True)[0][:, 0:1]
    # Divide by M*G (~2207 N) to normalise to dimensionless O(1) residual
    R_v     = (M_TOTAL * dv_dt + outs["mu_eff"] * M_TOTAL * G) / (M_TOTAL * G)
    R_theta = dtheta_dt - outs["v"] * torch.sin(outs["theta"]) / WHEELBASE
    return R_v, R_theta

def residual_biomechanics(outs, x):
    """
    Kelvin-Voigt brain ODE (3 axes):
        M_b*x_b'' + C*x_b' + K*x_b = M_b * a_helmet
    Divide by K_BRAIN (~2.1e4 N/m) to normalise to dimensionless O(1) residual.
    """
    Rs = []
    for x_b, a_h in zip(
        [outs["x_brain"], outs["y_brain"], outs["z_brain"]],
        [x[:,1:2], x[:,2:3], x[:,3:4]]
    ):
        x_b_t  = grad(x_b.sum(),   x, create_graph=True)[0][:, 0:1]
        x_b_tt = grad(x_b_t.sum(), x, create_graph=True)[0][:, 0:1]
        Rs.append((M_BRAIN*x_b_tt + C_BRAIN*x_b_t + K_BRAIN*x_b - M_BRAIN*a_h) / K_BRAIN)
    return Rs

def pinn_loss(model, x_batch, y_batch, lam):
    x_batch = x_batch.clone().requires_grad_(True)
    outs    = model.get_outputs(x_batch)
    pred    = torch.cat([outs["v"], outs["theta"], outs["mu_eff"]], dim=1)
    L_d     = nn.functional.mse_loss(pred, y_batch)
    R_v, R_th = residual_vehicle(outs, x_batch)
    L_veh   = (R_v**2).mean() + (R_th**2).mean()
    Rs_bio  = residual_biomechanics(outs, x_batch)
    L_bio   = sum((R**2).mean() for R in Rs_bio)
    L_total = lam["data"]*L_d + lam["vehicle"]*L_veh + lam["bio"]*L_bio
    return L_total, {"total":L_total.item(),"data":L_d.item(),
                     "vehicle":L_veh.item(),"bio":L_bio.item()}

print("Loss functions ready.")

In [12]:
# Training Loop

def train(model, X_raw, Y_raw,
          epochs=1500, lr=1e-3, batch=512,
          lam=None, log_every=300):

    if lam is None: lam = {"data":1.0,"vehicle":0.1,"bio":0.05}

    norm = IMUNormaliser()
    X_n  = norm.fit_transform(X_raw)
    Ym   = Y_raw.mean(0); Ys = Y_raw.std(0)+1e-8
    Y_n  = (Y_raw-Ym)/Ys

    X_t = torch.tensor(X_n, dtype=torch.float32).to(DEVICE)
    Y_t = torch.tensor(Y_n, dtype=torch.float32).to(DEVICE)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_t,Y_t), batch_size=batch, shuffle=True)

    opt   = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    hist  = {k:[] for k in ["total","data","vehicle","bio"]}

    print(f"Training {epochs} epochs | batch={batch} | {DEVICE}")
    print(f"lambda: data={lam['data']} vehicle={lam['vehicle']} bio={lam['bio']}")
    print("-"*60)
    t0 = time.time()

    for ep in range(1, epochs+1):
        model.train(); ep_l = {k:0.0 for k in hist}
        for xb, yb in loader:
            opt.zero_grad()
            loss, losses = pinn_loss(model, xb, yb, lam)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            for k in ep_l: ep_l[k] += losses[k]
        nb = len(loader)
        for k in hist: hist[k].append(ep_l[k]/nb)
        sched.step()
        if ep % log_every == 0 or ep == 1:
            print(f"Ep {ep:5d}/{epochs}  "
                  f"total={hist['total'][-1]:.5f}  "
                  f"data={hist['data'][-1]:.5f}  "
                  f"veh={hist['vehicle'][-1]:.5f}  "
                  f"bio={hist['bio'][-1]:.5f}  "
                  f"({time.time()-t0:.0f}s)")

    print("-"*60)
    print(f"Done in {time.time()-t0:.1f}s")
    return norm, (Ym, Ys), hist


model = RoadSoSPINN().to(DEVICE)
norm, (Ym, Ys), hist = train(
    model, X_raw, Y_raw,
    epochs=3000, lr=1e-3, batch=512,
    lam={"data":1.0,"vehicle":0.1,"bio":0.05},
    log_every=300)

torch.save({"model_state":model.state_dict(),
            "norm_mean":norm.mean,"norm_std":norm.std,
            "Y_mean":Ym,"Y_std":Ys}, "roadsos_pinn.pt")
print("Saved: roadsos_pinn.pt")

Training 3000 epochs | batch=512 | cuda
lambda: data=1.0 vehicle=0.1 bio=0.05
------------------------------------------------------------
Ep     1/3000  total=635513.31968  data=1.29784  veh=859112.12331  bio=10992015.92230  (1s)


KeyboardInterrupt: 

In [ ]:
# Training Loss Plot

fig, axes = plt.subplots(1,3,figsize=(15,4),facecolor="#0d0d0d")
fig.suptitle("RoadSoS PINN - Training Loss", color="white", fontsize=14)
eps = np.arange(1, len(hist["total"])+1)
for ax_, key, clr, title in zip(
    axes,
    ["data","vehicle","bio"],
    [DARK_ACCENT, WARN_ACCENT, SAFE_ACCENT],
    ["Data Loss (MSE)","Vehicle Physics Residual","Biomechanics Residual"]
):
    ax_.plot(eps, hist[key], color=clr, lw=1.5)
    ax_.set_facecolor("#0d0d0d"); ax_.set_title(title, color="white", fontsize=10)
    ax_.set_xlabel("Epoch", color="white"); ax_.set_ylabel("Loss", color="white")
    ax_.tick_params(colors="white"); ax_.grid(alpha=0.15); ax_.set_yscale("log")
plt.tight_layout()
plt.savefig("training_loss.png", dpi=150, bbox_inches="tight", facecolor="#0d0d0d")
plt.show()

In [ ]:
# Evaluate on all 3 scenarios

def run_inference(sim_fn, sim_kw, fs):
    t,ax,ay,az,gx,gy,gz,v,theta,mu,psk,sc = sim_fn(**sim_kw)
    X = np.c_[t,ax,ay,az,gx,gy,gz]
    X_t = torch.tensor(norm.transform(X), dtype=torch.float32).to(DEVICE)
    model.eval()
    with torch.no_grad(): outs = model.get_outputs(X_t)
    mu_p  = outs["mu_eff"].cpu().numpy().flatten()
    P_sk  = np.clip((MU_WET - mu_p)/MU_WET, 0, 1)
    a_res = np.sqrt(ax**2+ay**2+az**2)/G
    hic,_ = compute_hic15(ax,ay,az,fs)
    bric_ = compute_bric(gx,gy,gz)
    lbl,clr,score = injury_label(hic,bric_)
    return dict(t=t,ax=ax,ay=ay,az=az,gx=gx,gy=gy,gz=gz,
                mu_true=mu,mu_pred=mu_p,P_skid=P_sk,a_res=a_res,
                hic=hic,bric=bric_,label=lbl,color=clr,score=score)

res_n = run_inference(simulate_normal,    {"duration":10.0,"seed":99}, FS_NORMAL)
res_o = run_inference(simulate_oil_patch, {"duration":10.0,"seed":99}, FS_NORMAL)
res_c = run_inference(simulate_crash,     {"duration":2.0, "seed":99}, FS_CRASH)

for name,r in [("Normal",res_n),("Oil patch",res_o),("Crash",res_c)]:
    print(f"{name:10s}  HIC15={r['hic']:6.0f}  BrIC={r['bric']:.3f}  "
          f"P_skid_max={r['P_skid'].max():.2f}  [{r['label']}]")

In [ ]:
# Full Dashboard Plot (dark mode)

fig = plt.figure(figsize=(18,13), facecolor="#0a0a0a")
fig.suptitle("RoadSoS - PINN Evaluation Dashboard",
             color=DARK_ACCENT, fontsize=17, fontweight="bold")
gs = gridspec.GridSpec(3,3,figure=fig,hspace=0.55,wspace=0.38)

for row,(r,sc_lbl,sc_clr) in enumerate([
    (res_n,"Normal Riding",SAFE_ACCENT),
    (res_o,"Oil Patch",    WARN_ACCENT),
    (res_c,"Crash",        CRIT_ACCENT),
]):
    # Acceleration
    a0 = fig.add_subplot(gs[row,0])
    a0.plot(r["t"],r["a_res"],color=sc_clr,lw=1.2)
    a0.axhline(6,color=CRIT_ACCENT,ls="--",lw=0.8,label="6g")
    a0.set_title(f"{sc_lbl} - Accel",color=sc_clr,fontsize=9)
    a0.set_ylabel("g",color="white"); a0.tick_params(colors="white")
    a0.grid(alpha=0.15); a0.set_facecolor("#0a0a0a")
    if row==0: a0.legend(fontsize=7,labelcolor="white")

    # Skid probability
    a1 = fig.add_subplot(gs[row,1])
    a1.fill_between(r["t"],r["P_skid"],alpha=0.35,color=sc_clr)
    a1.plot(r["t"],r["P_skid"],color=sc_clr,lw=1.5)
    a1.axhline(0.65,color=WARN_ACCENT,ls="--",lw=0.8,label="Alert 0.65")
    a1.set_ylim(0,1.05); a1.set_title("PINN P(skid)",color=sc_clr,fontsize=9)
    a1.set_ylabel("Probability",color="white"); a1.tick_params(colors="white")
    a1.grid(alpha=0.15); a1.set_facecolor("#0a0a0a")
    if row==0: a1.legend(fontsize=7,labelcolor="white")

    # Scorecard
    a2 = fig.add_subplot(gs[row,2]); a2.axis("off"); a2.set_facecolor("#0a0a0a")
    hic_c  = CRIT_ACCENT if r["hic"]>HIC_HIGH  else(WARN_ACCENT if r["hic"]>HIC_LOW else SAFE_ACCENT)
    bric_c = CRIT_ACCENT if r["bric"]>1.0       else(WARN_ACCENT if r["bric"]>0.6   else SAFE_ACCENT)
    for txt,yp,col,fs_ in [
        (sc_lbl,                    0.88, sc_clr,    12),
        (f"HIC15 = {r['hic']:.0f}", 0.68, hic_c,    15),
        (f"BrIC  = {r['bric']:.3f}",0.48, bric_c,   15),
        (f"[ {r['label']} RISK ]",  0.26, r["color"],13),
    ]:
        a2.text(0.5,yp,txt,ha="center",va="center",
                transform=a2.transAxes,color=col,fontsize=fs_,fontweight="bold")

plt.savefig("pinn_dashboard.png",dpi=150,bbox_inches="tight",facecolor="#0a0a0a")
plt.show()
print("Saved: pinn_dashboard.png")

# StreamLit Dashboard Code

In [ ]:
%%writefile roadsos_dashboard.py
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

st.set_page_config(page_title="RoadSoS", page_icon="🪖", layout="wide")
plt.style.use("dark_background")

G=9.81; M_TOTAL=225; WHEELBASE=1.35; M_BRAIN=1.4; K_BRAIN=2.1e4; C_BRAIN=85
MU_DRY=0.75; MU_WET=0.45; MU_OIL=0.15
BRIC_X=66.3; BRIC_Y=56.5; BRIC_Z=42.2; HIC_LOW=700; HIC_HIGH=1000
FS_N=100; FS_C=1000
CA="#00E5FF"; CW="#FFD600"; CR="#FF1744"; CG="#00E676"; BG="#0d0d0d"


class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        d = [7]+[128]*6+[6]
        layers = []
        for i in range(len(d)-1):
            layers.append(nn.Linear(d[i], d[i+1]))
            if i < len(d)-2:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def out(self, x):
        o = self.forward(x)
        return {
            "v": o[:,0:1], "theta": o[:,1:2],
            "mu_eff": torch.sigmoid(o[:,2:3]),
            "xb": o[:,3:4], "yb": o[:,4:5], "zb": o[:,5:6]
        }


@st.cache_resource
def load_model():
    ck = torch.load("roadsos_pinn.pt", map_location="cpu")
    m = PINN()
    m.load_state_dict(ck["model_state"])
    m.eval()
    return m, ck["norm_mean"], ck["norm_std"]


def hic15(ax, ay, az, fs, wms=15):
    ar = np.sqrt(ax**2+ay**2+az**2)/G
    dt = 1/fs; mw = max(2, int(wms*1e-3/dt)); h = 0
    cs = np.cumsum(ar)  # computed once outside loop
    for w in range(2, mw+1):
        ms = (cs[w:]-cs[:-w])/w
        vals = w*dt*(np.maximum(ms,0)**2.5)
        h = max(h, vals.max())
    return h


def bric(gx, gy, gz):
    return np.sqrt(
        (np.max(np.abs(gx))/BRIC_X)**2 +
        (np.max(np.abs(gy))/BRIC_Y)**2 +
        (np.max(np.abs(gz))/BRIC_Z)**2)


def sim(sc, dur, fs, seed=42):
    np.random.seed(seed)
    t = np.linspace(0, dur, int(dur*fs)); n = len(t)
    if sc == 0:
        v = 11.1+0.5*np.sin(0.3*t)+0.1*np.random.randn(n)
        th = 0.05*np.sin(0.2*t)+0.01*np.random.randn(n)
        mu = np.clip(MU_DRY+0.02*np.random.randn(n), 0.6, 0.9)
    elif sc == 1:
        pi = int(0.4*n); v = 13.9+0.8*np.sin(0.3*t)
        mu = np.ones(n)*MU_DRY
        dec = np.exp(-np.arange(n-pi)/(0.2*fs))
        mu[pi:] = MU_OIL+(MU_DRY-MU_OIL)*dec[:n-pi]
        mu = np.clip(mu, 0.05, 0.9)
        th = 0.05*np.sin(0.2*t); th[pi:] += 0.3*(1-dec[:n-pi])
    else:
        ii = int(0.25*n); sw = int(0.1*fs); v = np.ones(n)*16.7
        v[ii:ii+sw] = np.linspace(16.7,0,sw); v[ii+sw:] = 0
        th = np.zeros(n)
        th[ii:] = np.clip(np.linspace(0,np.pi/2*1.2,n-ii), 0, np.pi/2)
        mu = np.ones(n)*MU_DRY; mu[ii:] = MU_OIL
    ax = np.diff(v,prepend=v[0])*fs+0.1*np.random.randn(n)
    ay = v*np.gradient(th,1/fs)+0.1*np.random.randn(n)
    az = G*np.cos(th)+0.05*np.random.randn(n)
    gx = np.gradient(th,1/fs)+0.02*np.random.randn(n)
    gy = 0.05*np.sin(0.1*t)+0.01*np.random.randn(n)
    gz = 0.03*np.cos(0.15*t)+0.01*np.random.randn(n)
    if sc == 2:
        pw = int(0.01*fs); ii2 = int(0.25*n); pulse = np.zeros(n)
        if ii2+pw < n:
            pulse[ii2:ii2+pw] = 10*G*np.sin(np.linspace(0,np.pi,pw))
            gx[ii2:ii2+pw] += 25*np.sin(np.linspace(0,np.pi,pw))
        ax += pulse
    return t, ax, ay, az, gx, gy, gz, v, th, mu


# Sidebar
st.sidebar.title("RoadSoS Controls")
sc_name = st.sidebar.selectbox("Scenario", ["Normal Riding","Oil Patch","Crash"])
sc = {"Normal Riding":0, "Oil Patch":1, "Crash":2}[sc_name]
fs = FS_C if sc==2 else FS_N
dur = st.sidebar.slider("Duration (s)", 1.0, 15.0, 8.0 if sc<2 else 2.0, 0.5)
st.sidebar.markdown("---")
st.sidebar.markdown("**Rider Profile**")
bg  = st.sidebar.selectbox("Blood Group",["O+","O-","A+","A-","B+","B-","AB+","AB-"])
alg = st.sidebar.text_input("Allergies","None")
ec  = st.sidebar.text_input("Emergency Contact","+91-XXXXXXXXXX")

# Header
st.markdown(f"<h1 style='color:{CA};font-family:monospace;'>RoadSoS Live Monitor</h1>", unsafe_allow_html=True)
st.markdown("<p style='color:gray;'>PINN | HIC15 | BrIC | Kelvin-Voigt Brain Model | Tire-Road ODE</p>", unsafe_allow_html=True)

# Inference
model, nm, ns = load_model()
t, ax, ay, az, gx, gy, gz, v, th, mu = sim(sc, dur, fs)
X = np.c_[t,ax,ay,az,gx,gy,gz]
Xn = (X - nm) / (ns + 1e-8)
Xt = torch.tensor(Xn, dtype=torch.float32)
with torch.no_grad():
    o = model.out(Xt)
mu_p = o["mu_eff"].numpy().flatten()
Psk  = np.clip((MU_WET-mu_p)/MU_WET, 0, 1)
ares = np.sqrt(ax**2+ay**2+az**2)/G
hv   = hic15(ax, ay, az, fs)
bv   = bric(gx, gy, gz)
inj  = min(0.6*hv/HIC_HIGH + 0.4*bv, 1.0)

# Alert banner
crash_det = (sc==2 or ares.max()>6)
skid_det  = (Psk.max()>0.65 and not crash_det)
ac = CR if crash_det else (CW if skid_det else CG)
at = "CRASH DETECTED - SOS FIRED" if crash_det else ("SKID WARNING - HAPTIC ALERT" if skid_det else "SAFE")
st.markdown(f"<h2 style='color:{ac};'>{at}</h2>", unsafe_allow_html=True)

# Metrics
c1,c2,c3,c4,c5 = st.columns(5)
c1.metric("Peak Accel",    f"{ares.max():.1f} g", ">6g" if ares.max()>6  else "OK")
c2.metric("P(skid) max",   f"{Psk.max():.2f}",   "HIGH" if Psk.max()>0.65 else "LOW")
c3.metric("HIC15",         f"{hv:.0f}",           "DANGER" if hv>700 else "OK")
c4.metric("BrIC",          f"{bv:.3f}",           "CRITICAL" if bv>1  else "OK")
c5.metric("Injury Score",  f"{inj:.2f}",          "SEVERE" if inj>0.7 else ("MOD" if inj>0.3 else "LOW"))
st.markdown("---")

# Plots
cl, cr_ = st.columns(2)
with cl:
    fig,(a1,a2) = plt.subplots(2,1,figsize=(8,6),facecolor=BG)
    a1.plot(t,ares,color=CA,lw=1.2); a1.axhline(6,color=CR,ls="--",lw=0.8,label="6g")
    a1.set_title("IMU Acceleration",color="white"); a1.set_ylabel("g",color="white")
    a1.tick_params(colors="white"); a1.grid(alpha=0.15); a1.set_facecolor(BG)
    a1.legend(labelcolor="white",fontsize=8)
    a2.fill_between(t,Psk,alpha=0.35,color=CW); a2.plot(t,Psk,color=CW,lw=1.5)
    a2.axhline(0.65,color=CR,ls="--",lw=0.8,label="Alert 0.65")
    a2.set_ylim(0,1.05); a2.set_title("PINN P(skid)",color="white")
    a2.set_ylabel("Probability",color="white"); a2.set_xlabel("Time (s)",color="white")
    a2.tick_params(colors="white"); a2.grid(alpha=0.15); a2.set_facecolor(BG)
    a2.legend(labelcolor="white",fontsize=8)
    plt.tight_layout(); st.pyplot(fig)

with cr_:
    fig2,(a3,a4) = plt.subplots(2,1,figsize=(8,6),facecolor=BG)
    a3.plot(t,mu_p,color=CG,lw=1.5,label="PINN mu_eff")
    a3.plot(t,mu,color="gray",lw=1,ls="--",alpha=0.6,label="True mu")
    a3.axhline(MU_WET,color=CW,ls=":",lw=0.8,label=f"Wet {MU_WET}")
    a3.axhline(MU_OIL,color=CR,ls=":",lw=0.8,label=f"Oil {MU_OIL}")
    a3.set_title("Friction Coefficient (PINN)",color="white"); a3.set_ylabel("mu_eff",color="white")
    a3.tick_params(colors="white"); a3.grid(alpha=0.15); a3.set_facecolor(BG)
    a3.legend(labelcolor="white",fontsize=7)
    a4.plot(t,np.abs(gx),color=CA,lw=1,label="|wx| roll")
    a4.plot(t,np.abs(gy),color=CW,lw=1,label="|wy| pitch")
    a4.plot(t,np.abs(gz),color=CG,lw=1,label="|wz| yaw")
    a4.set_title("Angular Velocity (Gyro)",color="white"); a4.set_ylabel("rad/s",color="white")
    a4.set_xlabel("Time (s)",color="white"); a4.tick_params(colors="white")
    a4.grid(alpha=0.15); a4.set_facecolor(BG); a4.legend(labelcolor="white",fontsize=7)
    plt.tight_layout(); st.pyplot(fig2)

# Rider card
st.markdown("---")
st.markdown(
    f"<div style='background:#111;padding:16px;border-radius:10px;border:1px solid {CA};'>"
    f"<h4 style='color:{CA};'>Rider Medical Profile - Transmitted on SOS</h4>"
    f"<p style='color:white;'>Blood: <b>{bg}</b> | Allergies: <b>{alg}</b> | Emergency: <b>{ec}</b></p>"
    f"<p style='color:gray;font-size:12px;'>Encrypted on ESP32-S3. Released via NFC/QR on crash detection.</p>"
    f"</div>",
    unsafe_allow_html=True)

In [ ]:
import os
from pyngrok import ngrok
import subprocess, time

proc = subprocess.Popen(["streamlit","run","roadsos_dashboard.py",
                          "--server.port","8501","--server.headless","true"])
time.sleep(4)
ngrok.set_auth_token(os.environ["NGROK_TOKEN"])  # set NGROK_TOKEN env var before running
url = ngrok.connect(8501)
print("Dashboard URL:", url)
print("Open the link above to view the live RoadSoS dashboard.")